<img src="../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../images/Konstanz_Logo.svg" width="200" /> <img src="../images/KIT_Logo.png" width="200" />

# Python Environments on the Cluster

How to create your own Python environment, make it available to Jupyter as a kernel, and
install the packages the later notebooks need.

> **This notebook explains the environment the whole workshop runs in.** Section 1 shows
> how virtual environments work in general, and how a kernel is registered so that
> Jupyter can use one. Section 2 then creates the workshop's own environment with `uv`,
> which is what [`09_Dask`](../09_Dask/02_Dask_On_The_Cluster.ipynb) rebuilds on every
> compute node it reserves.

---

## Contents

1. [Creating Environments](#creating-environments)
    - [Environments and Jupyter](#environments-and-jupyter)
2. [The environment for this workshop](#environment-for-the-following-exercises)
    - [Installing uv](#installing-uv)
    - [Creating the environment](#creating-the-environment)
    - [Registering the kernel](#registering-the-kernel)
    - [Checking the environment](#checking-the-environment)
    - [Adding a package](#adding-a-package)
3. [Remarks on Jupyter](#remarks-on-jupyter)

<a id="creating-environments"></a>
## 1. Creating Environments

__IMPORTANT: to create environments from within Jupyter or in order to switch into these environments, please pay attention to [Environments and Jupyter](#environments-and-jupyter)__

Python applications and modules often require further modules. These dependencies sometimes are bound to certain versions, e.g. Module A requires version 1 of Module C, however Module B requires C, however version 2. This creates a version conflict which needs to be resolved.

In order not having to re-install module versions back an forth globally, Virtual Environments allows storing projects with their corresponding dependencies.

To create a new Virtual Environment, please execute the following commands in a Terminal: "File"->"New"->"Terminal":

```bash
python3 -m venv <Name of your project for this Virtual Environment>
```

Please use the Python version, which You want to employ within your Virtual Environment. So, if You want to use Python3 in your environment, please use as such.

Once created, the environment needs to be activated by issuing a Shell script in the environments directory:

```bash
source <Name of the Virtual Environment>/bin/activate
```

In the activated environment new modules may be installed / uninstalled without changing the global set of Python modules (usually in Your $HOME/.cache/pip/)

```bash
python3 -m pip install ipykernel
python3 -m pip uninstall ipykernel
```
The requested modules are stored in <Name of your project for this Virtual Environment>/lib/python/
As soon as you don't need the environment you may unload it using the command:

```bash
deactivate
```

To delete a environment completely, you may remove the directory into which it was created.

<a id="environments-and-jupyter"></a>
### 1.1 Environments and Jupyter

When creating the Virtual Environment from the console of the bwUniCluster-Jupyter-Instance, prior to that you have to remove the already loaded `jupyter/base`-module.

The module `jupyter/base`-module and all it's dependencies may be removed using the Software icon at the very left of the Jupyter Desktop by clicking "unload".

![title](../images/bwUniCluster_Jupyter_Unload_BaseModule.png)

Additionally the corresponding LMod module may need to be unloaded (only necessary, if Terminal is opened before unloading the base module):

```bash
module list
module unload jupyter/base/2023-10-10    # please adapt the name.
```

Now, after activating your new Virtual environment and install new Python modules using `pip install`. In order to use your own environment with Jupyter, you have to install the jupyter-kernel in the environment:

```bash
python3 -m pip install ipykernel
```

Furthermore, the kernel has to be registered for the usage out of Jupyter

```bash
python3 -m ipykernel install --user --name=<Name of the Virtual Environment>
```

Such a kernel may be used in Jupyter (maybe need to reload the Browser's tab to show the kernel). It will contain all the Python modules loaded into the Virtual Environment.

In Jupyter, the environment may be switched by switching between Kernels. In an opened Notebook, please use the Kernel-menu (on th top-right corner).

It contains the name of the currently selected kernel: ![title](../images/kernel_selection1.png)

Using this menu, you get a drop-down list of the currently available (and the corresponding environments). By selecting the desired kernels and confirming the selection, the environment for a single Notebook may be changed.

![title](../images/kernel_selection2.png)

![title](../images/kernel_selection3.png)

<a id="environment-for-the-following-exercises"></a>
## 2. The environment for this workshop

Section 1 built an environment by hand: create it, activate it, `pip install` what you
need. That is worth knowing, and it is the wrong way to hand an environment to thirty
workshop participants — because "what you need" is a moving target. `pip install pandas`
today and the same command in six months give you different versions, and a notebook
that worked for the author then fails for someone else now.

So the workshop's environment is not built by hand. It is **described in the repository**
and created from that description with [`uv`](https://docs.astral.sh/uv/). Two files do
the work:

- **`pyproject.toml`** lists what the workshop needs, loosely: `pandas>=2.2` and so on.
- **`uv.lock`** records the exact version of every package that was actually resolved,
  including packages you never asked for directly, with a checksum for each.

The lock file is the important one. It means everybody in the room gets byte-identical
packages — on the cluster, on a laptop, and in the project's continuous integration —
and it is also what lets [`09_Dask`](../09_Dask/02_Dask_On_The_Cluster.ipynb) rebuild the
same environment on each compute node in seconds, straight from a local cache, rather
than packing an environment up and copying it around.

__IMPORTANT: unload the `jupyter/base` module before you start, as described in
[Environments and Jupyter](#environments-and-jupyter).__

<a id="installing-uv"></a>
### 2.1 Installing uv

All commands in this section go into a terminal (*File* &rarr; *New* &rarr; *Terminal*).
`uv` is a single binary and installs into your `$HOME`, so it needs no administrator
rights and no module:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

It lands in `~/.local/bin/uv`. If the shell cannot find it afterwards, either open a new
terminal or add it to your path for this session:

```bash
export PATH="$HOME/.local/bin:$PATH"
uv --version
```

<a id="creating-the-environment"></a>
### 2.2 Creating the environment

From inside your clone of the workshop repository:

```bash
cd ~/workshop-parallel-jupyter      # please adapt to where you cloned it
uv sync --all-extras
```

That reads `pyproject.toml` and `uv.lock`, downloads whatever is not already cached, and
builds a `.venv` directory holding every package the notebooks import. It also installs
a suitable Python interpreter if the system does not have one new enough — the workshop
needs at least Python 3.11, because the deep learning chapter's versions of TensorFlow
and PyTorch require it.

`--all-extras` deserves a word. The two heavy chapters are declared as **optional
extras** in `pyproject.toml`:

| Extra  | Contents                                  | Needed by |
|--------|-------------------------------------------|-----------|
| `dl`   | TensorFlow, TensorBoard, PyTorch          | `08_Deep_Learning` |
| `dask` | Dask, distributed, dask-ml, graphviz      | `09_Dask` |
| `mpi`  | dask-mpi, mpi4py                          | `09_Dask`, cluster part |

`--all-extras` installs all of them, which is what you want for working through the
workshop. The split exists because a Dask job builds this environment on every compute
node it reserves, and `uv sync --extra dask --extra mpi` leaves out several GB of deep
learning frameworks that no Dask worker will ever import.

Note that you never activate this environment by hand. `uv run <command>` runs a command
inside it:

```bash
uv run python -c "import pandas; print(pandas.__version__)"
```

<a id="registering-the-kernel"></a>
### 2.3 Registering the kernel

For Jupyter to offer the environment as a kernel, it has to be registered — exactly as in
[section 1.1](#environments-and-jupyter), but through `uv run`:

```bash
uv run python -m ipykernel install --user --name workshop --display-name "Workshop"
```

Open any notebook and pick **Workshop** from the kernel selector in the top right corner.
If it does not appear, reload the browser tab.

<a id="checking-the-environment"></a>
### 2.4 Checking the environment

The repository carries a script that collects every `import` statement in every notebook
and verifies that the environment satisfies all of them:

```bash
uv run python scripts/test_environment.py
```

The same check runs automatically on GitHub whenever the notebooks or the dependencies
change, so a missing package is usually caught before anyone reaches the cluster.

<a id="adding-a-package"></a>
### 2.5 Adding a package

If you want a package that is not in the list, do not `pip install` it into `.venv` — the
next `uv sync` would remove it again, and nobody else would get it. Add it to the
description instead:

```bash
uv add <package>
```

This updates `pyproject.toml`, re-resolves `uv.lock` and installs the result, so the
change is reproducible for everyone who pulls the repository. `uv remove <package>`
reverses it.

<a id="remarks-on-jupyter"></a>
## 3. Remarks on Jupyter

Jupyter offers good interactivity with your code and data. The source code may be split in several cells and each and every cell executed, changed and re-executed again. Therefore Jupyter is a good tool to design and experiment with data. In order to enable this functionality, Jupyter requests and allocates resources from the Cluster's queueing system, even though they are hardly occupying the hardware.
Also with regard to Python execution, the possibility to amend code and data in the cells hinders Python's GarbageCollector to clean up state.
Therefore it is a good idea to keep to the smallest possible input data set and/or to delete data manually, as soon as it is no longer required:

In retrospect:
- Use Jupyter during the development process and for documenting/communicating ideas with others
- Reduce the data set size as much as possible
- Use the `del`-instruction to delete data not longer required
- Move the code developed in Jupyter into it's stand-alone Python-Script to work on large datasets